In [1]:
# 51. re.sub with a Callable Callback
# When doing complex string replacements where the replacement depends on 
# the match content, pass a function to re.sub instead of performing multiple passes.

import re

text = "Prices are $10 and $20"

# Increment every price found by 
def increment(match):
    val = int(match.group(1))
    return f"{val + 5}"

# Pass the function as the second argument
result = re.sub(r'\$(\d+)', increment, text)

print(result)   


Prices are 15 and 25


In [3]:
# 52. collections.UserList for Custom List Logic
# Inheriting directly from list works, but it's brittle because 
# built-in methods (like append) don't necessarily call your overridden methods.
# UserList is a wrapper class designed specifically for clean inheritance.

from collections import UserList

class SortedList(UserList):
    def append(self, item):
        # Custom logic: insert in sorted order instead of just appending
        # This is cleaner than overriding __init__ and handling list() internals
        self.data.append(item)
        self.data.sort()

sl = SortedList([3,1,2])
sl.append(0)
print(sl)  

[0, 1, 2, 3]


In [8]:
# 53. dataclasses.field(default_factory) for Mutable Defaults

# A classic Python trap is using a mutable default argument 
# (like def func(l=[])). In dataclasses, using default=[] errors out. 
# The idiomatic fix is default_factory.

from dataclasses import dataclass, field

@dataclass
class Graph:
    # If we use default=[], all instances share the SAME list!
    # default_factory creates a NEW list for each instance
    edges: list = field(default_factory=list)

g1 = Graph()
g1.edges.append(1)
g2 = Graph()

print(g1.edges, g2.edges)  # Output: g2: [] (Safe, empty list
print(id(g1), id(g2))   # different address

[1] []
4362602352 4362601056


In [10]:
# 54. types.MappingProxyType for Read-Only Dictionaries

# If you return a dictionary from a function but want to ensure the caller 
# doesn't modify the internal state, wrap it in a MappingProxyType. 
# It creates a read-only view.

from types import MappingProxyType

internal_config = {'host': 'localhost', 'port': 8080}
read_only_config = MappingProxyType(internal_config)

read_only_config['port'] = 90   # 'mappingproxy' object does not support item assignment


TypeError: 'mappingproxy' object does not support item assignment

In [24]:
# 55. math.fsum for precise float sums

# Standard sum() uses floating-point addition, which accumulates error (drift) 
# over many additions. math.fsum tracks partial sums to minimize error, 
# crucial for geometry or finance problems.

# IEEE 754 standard for floating-point arithmetic.
import math

nums = [0.1] * 11

print(sum(nums), f"{sum(nums):.20f}")   # Output: 0.9999999999999999 (Drift)
print(math.fsum(nums))  # Output: 1.0 (Precise)

1.1 1.10000000000000008882
1.1


In [27]:
# 56. Slice assignment for in-place list modification

# Python lists support slice assignment, allowing you to replace, remove, or 
# insert chunks of data in a single operation. 
# This is powerful for "array splice" algorithms.

arr = [1,2,3,4,5]

# Replace middle section 
arr[1:4] = [20, 30]
print(arr)

# Delete a chunk
arr[1:2] = []
print(arr)

# Insert at specific index (slice of length 0)
arr[1:1] = [99]
print(arr)

[1, 20, 30, 5]
[1, 30, 5]
[1, 99, 30, 5]


In [44]:
# 57. f"{var}" for self-documentating debugging (python 3.8+)

# Debugging complex algorithms often involves printing variables. 
# Instead of print(f"x={x}"), you can simply add = inside the f-string.

x = 10
y = 20

# Prints "x=10, y=20"
print(f"{x=}, {y=}")

# Work with expression too
print(f"{x+y=}")

x=10, y=20
x+y=30


In [46]:
# 58. int.bit_count() (Python 3.8+)

# The "Popcount" (counting set bits) is a staple in bit manipulation DP. 
#The old idiom was bin(n).count('1'), which involves string conversion. 
# bit_count() is a dedicated C-level method that is significantly faster.

n = 15  # binary 1111

# Old way
print(bin(n).count('1'))

# Idiomatic fast way
print(n.bit_count()) # Output: 4

4
4


In [47]:
# 59. textwrap.dedent for clean multi-line strings
# When constructing complex test cases or SQL queries in code, 
# indentation makes the string look messy (with leading spaces). 
# dedent removes the common leading whitespace.

import textwrap

# Without dedent, the string contains "   SELECT..."
query = """
    SELECT * 
    FROM users 
    WHERE id = 1
"""

# Clean string
clean_query = textwrap.dedent(query).strip()
print(clean_query)


SELECT * 
FROM users 
WHERE id = 1


In [50]:
# 60. sys.intern for memory optimization

# If your DSA problem involves millions of repeated strings 
# (e.g., parsing logs or DNA sequences), Python creates a new object for 
# each. sys.intern ensures that identical strings share the same memory reference, 
# drastically reducing memory usage and speeding up dictionary lookups 
# (since pointer comparison is faster than string comparison).

import sys

# Standard strings (different objects)
a = "hello_world"
b = "hello_world"
print(a is b) # True, python caches small strings automatically

# Long dynamic strings usually create new objects
s1 = "long_dynamic_string_123"
s2 = "".join(["long_dynamic_string_", "123"])
print(s1 is s2) # False

# Using intern forces them to share memory
s2 = sys.intern(s2)
print(s1 is s2) # False (s1 is not interned yet)

s1 = sys.intern(s1)
print(s1 is s2) # True (Now they are the same object)

True
False
True
True


In [53]:
# 61. typing.Literal for Strict Input Validation

# In DSA, sometimes a function accepts a parameter that can only be one of 
# a few specific strings (e.g., direction="LEFT"). 
# Literal makes this explicit and helps IDEs catch typos.

from typing import Literal

def move(direction: Literal["UP", "DOWN", "LEFT", "RIGHT"]):
    # Logic here
    pass

move("UP")  # ok
# move("DIAGONAL")  # IDE warning


In [54]:
# 62. __call__ for Stateful functions

# You can make a class instance callable by defining __call__. 
# This is useful for creating "functors" that maintain state 
# (like a memoization cache or a counter) but can be used like a standard function.

class Counter:
    def __init__(self):
        self.count = 0

    def __call__(self):
        self.count += 1
        return self.count

c = Counter()
print(c())
print(c())

1
2


In [57]:
# 63. functools.singleddispatch for Type-Based Logic

# If you are processing a list of mixed types and need different logic for 
# each type, singledispatch allows you to "overload" a function based on the 
# type of the first argument. This is cleaner than a large if isinstance chain.

from functools import singledispatch

@singledispatch
def process(value):
    raise NotImplementedError("Unsupported type")

@process.register
def _(value: int):
    return value * 2

@process.register
def _(value: str):
    return value.upper()

@process.register
def _(value: list):
    return sum(value)

print(process(0))
print(process("hello"))

0
HELLO


In [58]:
# 64. contextlib.redirect_stdout for Capturing Output

# If you need to test a function that prints results instead of returning them, 
# or capture standard output for processing, use this context manager.

import io
from contextlib import redirect_stdout

def noisy_function():
    print("Result: 42")

# Capture the print into a string buffer
f = io.StringIO()
with redirect_stdout(f):
    noisy_function()

output = f.getvalue()
print(f"Captured: {output.strip()}")

Captured: Result: 42


In [61]:
# 65. var(obj) for Dictionary access

# When debugging a custom Object (like a Tree Node or Graph Edge), 
# printing node often shows unhelpful memory addresses. vars() 
# returns the __dict__ of the object, allowing you to view all 
# attributes as a dictionary.

class Node: 
    def __init__(self, val, left=None):
        self.val = val
        self.left = left

n = Node(10, Node(5))

print(vars(n))
    

{'val': 10, 'left': <__main__.Node object at 0x10582e6f0>}


In [62]:
# 66. Custom exceptions for control flow

# In backtracking or deep recursive searches (like N-Queens or Sudoku), 
# exiting a deep recursion stack immediately is tricky. Instead of passing boolean
# flags up the stack, raise a custom exception. It acts like a "localized goto".

class SolutionFound(Exception):
    pass 

def solve(grid):
    try: 
        backtrack(grid, 0)
    except SolutionFound:
        return grid  # Immediately returns the solved grid

def backtrack(grid, pos):
    if pos == 81:
        raise SolutionFoound # Jump all the way up!
    # ... recursive logic ...        
    

In [64]:
# 67. The @ operator for matrix multiplication

# Python 3.5 introduced @ specifically for matrix multiplication. 
# If you implement a Matrix class or use libraries like numpy, 
# this operator makes the code mathematically readable. 
# You can implement __matmul__ in your own classes.

class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __matmul__(self, other):
        # Example: Dot product
        return self.x * other.x + self.y * other.y

v1 = Vector(2, 3)
v2 = Vector(4, 5)

# Clean syntax for dot product
print(v1 @ v2)

23


In [67]:
# 68. statistics module for basic analysis

# Don't write your own mean/median functions. The statistics module provides 
# high-level helpers that are clear and correct.

import statistics 
data = [3,1,4,1,5,9]
print(statistics.mean(data))     # arithmetic mean of data  
print(statistics.mode(data))
print(statistics.median(data))   # middle value

3.8333333333333335
1
3.5


In [70]:
# 69. int.from_bytes for bit packing

# When working with low-level data or custom hashing, 
# you might need to convert a bytes object into an integer.

data = b'\x00\x10'   # 2 bytes

# Convert bytes to integer (Big Endian)
val = int.from_bytes(data, byteorder='big')
print(val)

16


In [71]:
# 70. breakpoints() for Modern Debugging (python 3.7+)

# Instead of inserting import pdb; pdb.set_trace(), simply use breakpoint(). 
# It drops you into an interactive debugger at that line, 
# allowing you to inspect variables and step through code.

def complex_calculation(x):
    y  = x * 2
    breakpoing()   # Execution pauses here
    return y / 0

In [72]:
# 71. Nested tuple unpacking in loops

# If you have a list of tuples within tuples (common in graph edges
# like (u, (v, w))), you can unpack them directly in the for statement
# using parentheses.

edges = [
    (1, (2, 10)),  # Node 1 connects to Node 2 with weight 10
    (1, (2, 3))
]

# Unpack three variables deep
for u, (v, w) in edges:
    print(f"Edge {u} -> {v} costs {w}")

Edge 1 -> 2 costs 10
Edge 1 -> 2 costs 3


In [74]:
# 72. str.casefold for Robust String Matching

# When comparing strings case-insensitively (e.g., anagrams or palindromes), 
# .lower() is usually used. However, .casefold() is stronger and handles 
# international characters correctly (like German 'ß' vs 'ss').

s1 = "straße"
s2 = "STRASSE"

# .lower() fails because 'ß'.lower() is 'ß'
print(s1.lower() == s2.lower())  # False

# .casefold() converts 'ß' to 'ss'
print(s1.casefold() == s2.casefold())  # Output: True


False
True


In [77]:
# 73. In-place set operations

# Python sets support in-place modification using operators like |=, &=, and -=. 
# This is more memory efficient than creating new set objects.

a = {1,2,3,4}
b = {3,4,5}

# Intersection Update: Keep only elements found in both
a &= b  # Equivalent to a.intersection_update(b)
print(a)

# Difference update: Remove elements found in b
a -= {3}
print(a)  # Remove {4}

{3, 4}
{4}


In [80]:
# 74. math.hypot for Euclidean Distance

# Calculating distance usually involves sqrt(x**2 + y**2). 
# math.hypot does this cleaner and avoids intermediate overflow/underflow issues.

import math

x1, y1 = 0, 0
x2, y2 = 3, 4

# Distance betwen points
dist = math.hypot(x2 - x1, y2 - y1)
print(dist)  # Output: 5.0

# It also supports n-dimensions in newer Python versions
dist_3d = math.hypot(1,2,2)
print(dist_3d)    

5.0
3.0


In [81]:
# 75. enumerate with a start Index

# Common in DSA problems involving 1-based indexing (like heaps or 
# specific grid problems). Instead of i + 1, use the start argument.

arr = ['a', 'b', 'c']

# 1-based indexing
for i, val in enumerate(arr, start=1):
    print(f"{i}: {val}")


1: a
2: b
3: c


In [82]:
# 76. itertools.tee for Multiple Iterators

# Generators can only be consumed once. If you need to iterate over the 
# same generator twice (e.g., once for max, once for min), 
# use tee to create independent copies.

from itertools import tee, islice

def gen():
    yield 1; yield 2; yield 3

# create two independent iterators from one generator
it1, it2 = tee(gen())

print(list(it1))
print(list(it2))
    

[1, 2, 3]
[1, 2, 3]


In [86]:
# 77. json for Fast Deep Copying

# While copy.deepcopy() is robust, it can be slow for large nested structures. 
# If your data consists only of JSON-serializable types (dicts, lists, strings, 
# ints), the "hacky" way to deep copy is often faster: serialize to string 
# and parse back.

import json

matrix = [[1,2], [3,4]]

# Fast deep copy for simple structure
matrix_copy = json.loads(json.dumps(matrix))

matrix_copy[0][0] = 99  # original is safe
print(matrix[0][0], matrix, matrix_copy)

1 [[1, 2], [3, 4]] [[99, 2], [3, 4]]


In [92]:
# 78. sys.getsizeof for memory profiling

# In strict memory limit environments, it helps to know exactly 
# how much memory your data structures consume. 
# sys.getsizeof returns the size in bytes.

import sys

# Integer overhead 
print(sys.getsizeof(1))

# List overhead
print(sys.getsizeof([]))

# Compare to array (much smaller)
from array import array
print(sys.getsizeof(array('i', [1])))  ## (scales better for many items)

28
56
84


In [93]:
# 79. deque.extendleft for reverse building

# extendleft adds elements from an iterable to the left side of the deque. 
# A crucial detail is that the order of the input iterable is reversed 
# in the process. This makes it ideal for building a deque in reverse 
# order efficiently.

from collections import deque

d = deque([3,4])
d.appendleft(2)
d.appendleft(1)
d.extendleft([0])
print(d)

deque([0, 1, 2, 3, 4])


In [95]:
# 80. math.copysign for Geometry / Physics

# In vector math or geometry problems, you might need to transfer 
# the sign of one number to the magnitude 
# of another without messy if statements

import math

x = 5
y = -2

# Result takes magnitude of x (5) and sign of y (-)
result = math.copysign(x, y)   # value of x with sign of y
print(result)

-5.0


In [99]:
# 81. re.finditer for Memory-Efficient Parsing

# re.findall returns a list of all matches, which consumes memory for large strings. 
# re.finditer returns an iterator, yielding match objects one by one.

import re

text = "Token 1: A, Token 2: B, Token: C"

# findall creates a full list in memory
matches = re.findall(r"Token \d", text)
print(matches)

# finditem is lazy (better for large text)
for match in re.finditer(r'Token (\d): (\w)', text):
    print(f"Found ID: {match.group(1)}, Val: {match.group(2)}")

['Token 1', 'Token 2']
Found ID: 1, Val: A
Found ID: 2, Val: B


In [101]:
# 82. list.clear() for In-Place Clearing

# When reusing a list (e.g., resetting a visited buffer or clearing a level in BFS), 
# use .clear() is more readable and efficient than del list[:] or assigning 
# a new empty list[]

buffer = [1,2,3]
print(buffer)
buffer.clear()  # clear content, keep object reference
print(buffer)

[1, 2, 3]
[]


In [102]:
# 83. str.removeprefix and removesuffic (python 3.9+)

# Parsing strings often involves removing a specific 
# start or end. Before 3.9, this required manual slicing with length checks. 
# These new methods are safer and cleaner.

filename = "data_final.csv"

# Remove prefix if it exists
name = filename.removeprefix("data_")

# Remove suffix if it exists
clean = name.removesuffix(".csv")
print(name, clean)

final.csv final


In [104]:
# 84. itertools.filterfalse for Inverse Filtering

# Sometimes you want to keep elements that fail a condition. 
# While you can use a list comprehension with if not ..., 
# filterfalse is explicit and works lazily on iterators.

from itertools import filterfalse

nums = [1,2,3,4,5,6]

# Keep only odd number
odds = list(filterfalse(lambda x: x % 2 == 0, nums))
print(odds)

# equivalent to 
odds = list(filter(lambda x: x % 2 != 0, nums))
print(odds)

[1, 3, 5]
[1, 3, 5]


In [106]:
# 85. bytearray for Multiple Strings

# Strings in Python are immutable; "modifying" one creates a new copy. 
# For problems requiring in-place character manipulation (like replacing 
# characters in a specific range), use bytearray. It behaves like a list 
# of integers (0-255) but is stored compactly.

s = "python"
# Strings are immutable: s[0] = 'P' -> Error

# Convert to mutable bytearray
b = bytearray(s, 'utf-8')
b[0] = 80 # ASCII for 'P'
b.extend(b' code')
print(b.decode())


Python code


In [108]:
# 86. heapq.heappushhop for Optimized Top-K

# In "Top K" problems, you often push an item and then pop the smallest. 
# Doing this in two steps (heappush then heappop) is 
# O(2logN). heappushpop does it in one efficient step.
    
import heapq

# Find the 3 largest numbers in a stream
k = 3
stream = [10, 1, 5, 20, 2, 100]
heap = []

for num in stream:
    print(heap)
    if len(heap) < k:
        heapq.heappush(heap, num)
    else:
        # If num is bigger than smallest in heap, sqap them
        if num > heap[0]:
            heapq.heappushpop(heap, num)
print(heap)

[]
[10]
[1, 10]
[1, 10, 5]
[5, 10, 20]
[5, 10, 20]
[10, 100, 20]


In [118]:
# 87. functools.cache for Simple Memorization (python 3.9+)

# While lru_cache is standard, Python 3.9 introduced @cache. 
# It is an unbounded cache (equivalent to lru_cache(maxsize=None)). 
# It is slightly faster and cleaner to write for pure DP problems 
# where you never want to evict entries.

from functools import cache

@cache
def fib(n):
    if n < 2: return n
    return fib(n-1) + fib(n-2)
    
print(fib(10))

55


In [119]:
# 88. types.SimpleNamespace for Dynamics Objects 

# If you need a lightweight object to hold data 
# (like a Node or State) but don't want to write a full class or use a dictionary 
# (which has quote overhead), use SimpleNamespace.

from types import SimpleNamespace

# Create a node dynamically
node = SimpleNamespace(val=10, left=None, right=None)

# Access attribute cleanly without ['key']
node.left = SimpleNamespace(val=5)

print(node.left.val)

5


In [123]:
# 89. str.splitlines() for Multi-line Input

# When reading input that contains multiple lines (like a grid), 
# using .split('\n') often leaves a trailing empty string if the input 
# ends with a newline. .splitlines() handles this edge case automatically.

raw_text = "Line 1\nLine 2\nLine 3\n"

print(raw_text.split('\n'))   # extra end ''
print(raw_text.splitlines())  # clean

['Line 1', 'Line 2', 'Line 3', '']
['Line 1', 'Line 2', 'Line 3']


In [134]:
# 90. Counting Occurences with bisect

# If you have a sorted list, you can count how many times a value 
# appears in O(logN) time using bisect_left and bisect_right. 
# This is faster than list.count() which is O(N)

import bisect
nums = [1,2,2,2,3,4]
target = 2

# Index of first 2 vs Index of element after last 2
count = bisect.bisect_right(nums, target) - bisect.bisect_left(nums, target)
print(count, bisect.bisect_right(nums, target), bisect.bisect_left(nums, target))

3 4 1


In [138]:
# 91. pprint for Debugging Nested Structures

# When debugging graphs or trees, a standard print(dict) results 
# in an unreadable one-liner. The pprint module (pretty print) 
# formats nested structures automatically.

from pprint import pprint

complex_data = {'A': [1,2,3], 'B': {'inner': [10, 20], 'status': True}}
# Standard print is messy for large dicts
pprint(complex_data)
# Output formats it nicely with identation

{'A': [1, 2, 3], 'B': {'inner': [10, 20], 'status': True}}


In [139]:
# 92. set.discard for Safe Removal

# When removing an element from a set, .remove(x) throws a KeyError if 
# x is missing. .discard(x) removes it if present and does nothing if absent. 
# This makes code cleaner by removing the need for if x in set: checks.

visited = {1,2,3}

# Safe removal
visited.discard(5)  # No error, nothing happens
visited.discard(2)
print(visited)

{1, 3}


In [141]:
# 93. min() and max() with default Argument

# A common DSA edge case is finding the min/max of an empty list, 
# which throws a ValueError. Python allows you to provide a default 
# return value for empty iterables.

nums = []

# min(nums) would crash
result = min(nums, default=0)  # returns 0 safely
print(result)

0


In [142]:
# 94. Counter.most_common(k) for Top-K elements

# Finding the top K frequent elements is a classic interview question. 
# Counter has a built-in method that returns the k most common 
# elements as a list of tuples.

from collections import Counter

arr = [1,1,1,2,2,3,4]

# find top 2 most frequent
top_2 = Counter(arr).most_common(2)

print(top_2)

[(1, 3), (2, 2)]


In [143]:
# 95. itertools.compress for Boolean Masking

# This is a hidden gem. Instead of using a list comprehension 
# with an if condition, compress filters an iterable based on a 
# corresponding boolean selector list.

from itertools import compress

data = ['a', 'b', 'c', 'd']
mask = [True, False, True, False]

result = list(compress(data, mask))
print(result)


['a', 'c']


In [144]:
# 96. Sorting by multiple Criteria (Mixed Ascending / Descending)

# Python's sort is stable and compares tuples element-by-element. 
# To sort by one field ascending and another descending, 
# you can invert the numeric value for the second field in the key.

users = [
    {'name': 'Alice', 'score': 50},
    {'name': 'Bob', 'score': 90},
    {'name': 'Charlie', 'score': 90}    
]

# Sort by Score (DESC), then Name (ASC)
users.sort(key=lambda x: (-x['score'], x['name']))

print([u['name'] for u in users])

['Bob', 'Charlie', 'Alice']


In [147]:
# 97. Unpacking inside print for Clean Output

# To print a list as space-separated values, the idiomatic way 
# is to unpack the list directly into the print function using 
# the * operator, rather than using a loop or join (if types differ).

nums = [1,2,3,4]

# Verbose way
print(' '.join(map(str, nums)))

# Idiomatic way
print(*nums)

# Customize the separator
print(*nums, sep=" | ")

1 2 3 4
1 2 3 4
1 | 2 | 3 | 4


In [152]:
# 98. itertools.repeat for Constant Iterators

# If you need to iterate a specific number of times 
# but don't care about the index, itertools.repeat is more efficient than range 
# because it doesn't need to generate and store numbers.

from itertools import repeat

# Repeat "Hello" 3 times
for _ in repeat(None, 3): # Use None as placeholder
    print("Action executed")

# Useful for filling arrays with constants
# e.g., creatng a matrix of all 5s
matrix = list(repeat([5,5], 3))
print(matrix)  # Output: 

Action executed
Action executed
Action executed
[[5, 5], [5, 5], [5, 5]]


In [157]:
# 99. dict.setdefault as a fallback

# While defaultdict is great, sometimes you are working 
# with a standard dict and don't want to convert it. setdefault(key, default) 
# returns the value if it exists; if not, it inserts the default and returns it.

# if value use it, if no value use default
graph = {}
# Manual way
if 'A' not in graph: 
    graph['A'] = []
print(graph)

# Idiomatic way
graph.setdefault('A', []).append(1)
print(graph)

{'A': []}
{'A': [1]}


In [158]:
# 100. try...except...else for Clean Flow

# This is a distinct Python idiom. The else block in a try statement 
# runs only if no exception occurred. This is useful for separating 
# the "happy path" code from the exception handling logic.

def safe_divide(a, b):
    try:
        result = a /b
    except ZeroDivisionError:
        print("Cannot divide by zero")
    else:
        # Only if division succeeded
        print(f'Result is {result}')
        return result

safe_divide(10, 2)

Result is 5.0


5.0